In [10]:
import pandas as pd
import numpy as np
df=pd.read_csv('Data_Cleaned_Silver_layer.csv')

In [19]:
# Mapping columns for the Star Dimension table
star_mapping = {
    'star_name': 'Star_Name',
    'st_teff(in kelvin)': 'Star Temperature',
    'st_rad(in solar radius)': 'Star Radius',
    'st_mass(in solar mass)': 'Stellar Mass',
    'st_lum(in Solar luminosity)': 'Stellar Luminosity',
    'sy_dist(in Parsec)': 'System_Distance',
    'sy_plx' : 'Parallax',
    'sy_snum': 'No_of_Stars'
}

# Drop dublicates for the dimension table
Dim_Star = df[list(star_mapping.keys())].drop_duplicates().reset_index(drop=True)

Dim_Star = Dim_Star.rename(columns=star_mapping)

Dim_Star['Is binary'] = Dim_Star['No_of_Stars'] > 1
Dim_Star['star_id'] = Dim_Star.index

# Display the resulting dimension table
display(Dim_Star.head())

,Star_Name,Star Temperature,Star Radius,Stellar Mass,Stellar Luminosity,System_Distance,Parallax,No_of_Stars,Is binary,star_id
0,11 Com,4874.0,13.76,2.09,95.866929,93.1846,10.71040,2,True,0
1,11 UMi,4213.0,29.79,2.78,250.839943,125.3210,7.95388,1,False,1
2,14 And,4888.0,11.55,1.78,68.324837,75.4392,13.22890,1,False,2
3,16 Cyg B,5750.0,1.13,1.08,1.252328,21.1397,47.27540,3,True,3
4,17 Sco,4157.0,25.92,1.22,180.003083,124.9530,7.98050,1,False,4


In [39]:
# Create Dim_Discovery dimension table
Dim_Discovery = df[['discoverymethod', 'disc_year']].drop_duplicates().reset_index(drop=True)
# Rename columns
Dim_Discovery = Dim_Discovery.rename(columns={
    'discoverymethod': 'Method',
    'disc_year': 'Year'
})

# Add a unique identifier for the dimension
Dim_Discovery['discovery_id'] = Dim_Discovery.index

display(Dim_Discovery.head())

,Method,Year,discovery_id
0,Radial Velocity,2007.0,0
1,Radial Velocity,2009.0,1
2,Radial Velocity,2008.0,2
3,Radial Velocity,1996.0,3
4,Radial Velocity,2020.0,4


In [21]:
# Define mapping for Planet dimension
planet_mapping = {
    'pl_name': 'PlanetName',
    'sy_pnum': 'Planet numbers',
    'pl_orbper': 'orbital period',
    'pl_orbsmax': 'Semi_major axis',
    'pl_orbeccen': 'Eccentricity',
    'pl_rade': 'Radius',
    'pl_bmasse': 'Mass',
    'pl_dens': 'Density',
    'pl_eqt': 'Temperature',
    'pl_ratror': 'Flux'
}

planet_cols = {
    'pl_name': 'PlanetName',
    'sy_pnum': 'Planet numbers',
    'pl_orb_period(in E.years)': 'orbital period',
    'pl_orb_semi_maj_axis(in AU)': 'Semi_major axis',
    'pl_orb_eccent': 'Eccentricity',
    'pl_rad(in Earth Radius)': 'Radius',
    'pl_bmass(in Earth Mass)': 'Mass',
    'pl_dens(in (Earth mass/Earth radius))': 'Density',
    'pl_teff(in kelvin)': 'Temperature',
    'pl_recei_flux(in Earth received flux)': 'Flux'
}

# Create Dim_Planet
Dim_Planet = df[list(planet_cols.keys())].drop_duplicates().reset_index(drop=True)
Dim_Planet = Dim_Planet.rename(columns=planet_cols)

Dim_Planet['planet_id'] = Dim_Planet.index

display(Dim_Planet.head())

,PlanetName,Planet numbers,orbital period,Semi_major axis,Eccentricity,Radius,Mass,Density,Temperature,Flux,planet_id
0,11 Com b,1,0.884901,1.178,0.238,3215.668419,4914.898486,8.140000e-07,803.218264,69.084121,0
1,11 UMi b,1,1.413333,1.530,0.080,3054.089914,4684.814200,9.060000e-07,896.380652,107.155343,1
2,14 And b,1,0.511321,0.775,0.000,662.607527,1131.151301,2.140000e-05,909.877272,113.756233,2
3,16 Cyg B b,1,2.186174,1.660,0.680,314.558722,565.737400,1.001520e-04,228.751790,0.454467,3
4,17 Sco b,1,1.583518,1.450,0.060,816.105731,1373.018718,1.390000e-05,847.471302,85.613832,4


In [22]:
Dim_Date = df[['disc_year']].drop_duplicates().reset_index(drop=True)
Dim_Date['Date_id'] = Dim_Date.index

display(Dim_Date.head())

,disc_year,Date_id
0,2007.0,0
1,2009.0,1
2,2008.0,2
3,1996.0,3
4,2020.0,4


In [34]:
# Creating the fact table: Fact_Exoplanets
# Create a base dataframe for merging to get the foreign keys
# This base should contain the unique combinations of the "natural keys" that link to dimensions
fact_base = df[['pl_name', 'star_name', 'discoverymethod', 'disc_year']].drop_duplicates().reset_index(drop=True)

# Merge with Dim_Planet to get planet_id
# We need to ensure that the 'pl_name' from fact_base matches 'PlanetName' in Dim_Planet
fact_exoplanets_temp = fact_base.merge(Dim_Planet[['PlanetName', 'planet_id']], left_on='pl_name', right_on='PlanetName', how='left')

# Merge with Dim_Star to get star_id
# 'star_name' from fact_base matches 'Star_Name' in Dim_Star
fact_exoplanets_temp = fact_exoplanets_temp.merge(Dim_Star[['Star_Name', 'star_id']], left_on='star_name', right_on='Star_Name', how='left')

# Merge with Dim_Discovery to get discovery_id
# 'discoverymethod', 'disc_year' from fact_base matches 'Method', 'Year' in Dim_Discovery
fact_exoplanets_temp = fact_exoplanets_temp.merge(Dim_Discovery[['Method', 'Year', 'discovery_id']], left_on=['discoverymethod', 'disc_year'], right_on=['Method', 'Year'], how='left')

# Merge with Dim_Date to get Date_id
# 'disc_year' from fact_base matches 'disc_year' in Dim_Date
fact_exoplanets_temp = fact_exoplanets_temp.merge(Dim_Date[['disc_year', 'Date_id']], left_on='disc_year', right_on='disc_year', how='left')

# Select only the foreign key columns for the final Fact_Exoplanets table
Fact_Exoplanets = fact_exoplanets_temp[['pl_name' ,'planet_id', 'star_name', 'star_id','discoverymethod', 'discovery_id','disc_year', 'Date_id']]

display(Fact_Exoplanets.head())

,pl_name,planet_id,star_name,star_id,discoverymethod,discovery_id,disc_year,Date_id
0,11 Com b,0,11 Com,0,Radial Velocity,0,2007.0,0
1,11 UMi b,1,11 UMi,1,Radial Velocity,1,2009.0,1
2,14 And b,2,14 And,2,Radial Velocity,2,2008.0,2
3,16 Cyg B b,3,16 Cyg B,3,Radial Velocity,3,1996.0,3
4,17 Sco b,4,17 Sco,4,Radial Velocity,4,2020.0,4


In [40]:
# 1. Save to Excel with multiple sheets
#------------------------------------------
with pd.ExcelWriter('Data_Gold_Layer.xlsx') as writer:
    Fact_Exoplanets.to_excel(writer, sheet_name='Fact_Exoplanets', index=False)
    Dim_Star.to_excel(writer, sheet_name='Dim_Star', index=False)
    Dim_Discovery.to_excel(writer, sheet_name='Dim_Discovery', index=False)
    Dim_Planet.to_excel(writer, sheet_name='Dim_Planet', index=False)
    Dim_Date.to_excel(writer, sheet_name='Dim_Date', index=False) # Added Dim_Date to excel export

# 2. Combine all for a single Gold Layer CSV
#----------------------------------------------
data_gold = Fact_Exoplanets.copy() # Fact_Exoplanets now contains planet_id, star_id, discovery_id, Date_id

# Merge Dim_Star to get star attributes
data_gold = data_gold.merge(Dim_Star, on='star_id', how='left')

# Merge Dim_Discovery to get discovery attributes
data_gold = data_gold.merge(Dim_Discovery, on='discovery_id', how='left')

# Merge Dim_Planet to get planet attributes
data_gold = data_gold.merge(Dim_Planet, on='planet_id', how='left')

# Merge Dim_Date to get date attributes
data_gold = data_gold.merge(Dim_Date, on='Date_id', how='left')

# Save the combined dataframe to CSV
data_gold.to_csv('Data_Gold_Layer.csv', index=False)

print("Files 'Data_Gold_Layer.csv' and 'Data_Gold_Layer.xlsx' have been created successfully.")

Files 'Data_Gold_Layer.csv' and 'Data_Gold_Layer.xlsx' have been created successfully.
